# 06 — Logistic Regression
First real model — with full evaluation on classification metrics.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from src.preprocess import build_preprocessor
from src.config import RANDOM_STATE

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

## 1. Train Logistic Regression Pipeline

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

## 2. Key Metrics

In [ ]:
f1 = f1_score(y_test, y_pred, pos_label=1)
accuracy = lr_pipeline.score(X_test, y_test)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 (class 1 — our primary metric): {f1:.4f}")
print(f"\nCompare vs Baseline: F1 went from 0.000 → {f1:.4f}")

## 3. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Default', 'Default'], ax=ax, cmap='Blues')
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=14, fontweight='bold')
plt.savefig('docs/eda_plots/lr_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Interpretation
- **False Negatives (FN):** Borrowers who defaulted but model predicted safe → financial loss
- **False Positives (FP):** Safe borrowers flagged as risky → lost business opportunity
- For lending, FN is more costly than FP
- Next: try class_weight and SMOTE to improve recall on defaulters